Notebook pre-requisites:

In [43]:
!pip install "camelot-py[cv]"
!pip install spacy


In [44]:
!python -m spacy download en_core_web_sm

  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [45]:
!pip install PyMuPDF
!pip install pdfplumber

## 1. PDF Ingestion & Parsing
- Extract text with page/section anchors (page number, heading hierarchy).
- Preserve structure: titles, subsections, lists, tables, figures’ captions.
- For tables: parse into machine-readable frames (CSV/JSON) when possible.
- Deliverables: raw_text.jsonl (chunks with metadata), tables/*.csv.

In [46]:
import fitz
import json
import re
from pathlib import Path
import shutil
import camelot
import warnings
import pandas as pd
import pdfplumber

def extract_tables(pdf_path, output_dir):
    output_dir = Path(output_dir)
    tables_dir = output_dir / "tables"
    tables_dir.mkdir(exist_ok=True)

    all_tables = []
    doc = fitz.open(pdf_path)

    for page_num in range(len(doc)):
        page_str = str(page_num + 1)
        page_tables = []

        # --- Try Camelot STREAM ---
        try:
            tables_stream = camelot.read_pdf(
                pdf_path, 
                pages=page_str, 
                flavor='stream',
                edge_tol=50,
                row_tol=10,
                strip_text='\n',
            )
            page_tables += [t.df for t in tables_stream if not t.df.empty]
        except Exception:
            pass

        # --- Try Camelot LATTICE ---
        if not page_tables:
            try:
                tables_lattice = camelot.read_pdf(
                    pdf_path, 
                    pages=page_str, 
                    flavor='lattice',
                    line_scale=40,
                    shift_text=['l', 't'],
                )
                page_tables += [t.df for t in tables_lattice if not t.df.empty]
            except Exception:
                pass

        # --- Fallback: pdfplumber ---
        if not page_tables:
            with pdfplumber.open(pdf_path) as pdf:
                page = pdf.pages[page_num]
                plumber_tables = page.extract_tables()
                for pt in plumber_tables:
                    df = pd.DataFrame(pt)
                    page_tables.append(df)

        # --- Clean and save ---
        for i, df in enumerate(page_tables):
            df = df.fillna("").astype(str).apply(lambda col: col.map(lambda x: re.sub(r"\n", " ", x).strip()))
            df = merge_split_rows(df)

            table_file = tables_dir / f"table_page{page_num+1}_{i+1}.csv"
            df.to_csv(table_file, index=False)

            all_tables.append({
                "table_number": i + 1,
                "page": page_num + 1,
                "file": str(table_file),
                "rows": df.shape[0],
                "columns": df.shape[1],
            })

    print(f"Extracted {len(all_tables)} tables total")
    print(f"   - Saved CSVs in {tables_dir}/")

    return all_tables


In [47]:
def merge_split_rows(df):
    df = df.fillna("")
    new_rows = []
    for idx, row in df.iterrows():
        if idx == 0:
            new_rows.append(row)
            continue
        if str(row[0]).strip() == "":
            # merge with previous row
            new_rows[-1] = new_rows[-1].combine(row, lambda x, y: f"{x} {y}".strip())
        else:
            new_rows.append(row)
    return pd.DataFrame(new_rows, columns=df.columns)

def extract_pdf_with_structure(pdf_path, output_dir="data"):
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    # prepare tables directory
    tables_dir = output_dir / "tables"
    if tables_dir.exists():
        shutil.rmtree(tables_dir)
    tables_dir.mkdir()

    # --- Text Extraction ---
    doc = fitz.open(pdf_path)
    chunks = []
    
    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text()

        # extract structure
        blocks = page.get_text("dict")["blocks"]
        
        # Detect headings (larger fonts)
        current_section = ""
        for block in blocks:
            if "lines" in block:
                for line in block["lines"]:
                    for span in line["spans"]:
                        if span["size"] > 12:  # Heading
                            current_section = span["text"]

        # split into chunks
        paragraphs = text.split("\n\n")
        for para in paragraphs:
            cleaned_para = para.strip().replace("\n", " ")
            if len(cleaned_para) > 50:
                chunks.append({
                    "page": page_num + 1,
                    "section": current_section,
                    "text": cleaned_para
                })
    
    # save cleaned text chunks
    clean_text_file = output_dir / "raw_text.jsonl"
    with open(clean_text_file, "w", encoding="utf-8") as f:
        for chunk in chunks:
            f.write(json.dumps(chunk, ensure_ascii=False) + "\n")
    
    print(f"Extracted {len(chunks)} text chunks")
    print(f"   - Clean text saved to {clean_text_file}")

    # --- Table extraction ---
    tables = extract_tables(pdf_path, output_dir)
        

    print(f"Extracted {len(tables)} tables")
    print(f"   - Tables saved as CSV in {tables_dir}/")
    
    return chunks, tables


In [48]:
chunks, tables = extract_pdf_with_structure("data/sustainable-health-from-food_web.pdf")

Extracted 120 text chunks
   - Clean text saved to data/raw_text.jsonl
Extracted 137 tables total
   - Saved CSVs in data/tables/
Extracted 137 tables
   - Tables saved as CSV in data/tables/
